# 第四章：符号特征提取

从 MIDI 文件中提取四类核心特征：

1. **音高直方图**（Pitch Histogram）
2. **音程向量**（Interval Histogram）
3. **节奏模式**（Rhythm Pattern）
4. **旋律轮廓**（Melodic Contour）

数据集：
- 茉莉花：`CODE/datasets/melodies/茉莉花.midi`
- 瑶族舞曲：`CODE/datasets/melodies/瑶族舞曲.midi`
- 可使用自己的 MIDI 文件进行分析

图片输出目录：`CODE/chapter04/output_figures/`（600 dpi）


In [ ]:
import os
import math
from collections import Counter

import matplotlib.pyplot as plt
import pretty_midi

# 中文字体：覆盖 Windows / macOS / Linux 常见环境
plt.rcParams['font.sans-serif'] = [
    'PingFang SC',        # macOS 默认中文
    'Hiragino Sans GB',   # macOS 备选
    'Microsoft YaHei',    # Windows 默认中文
    'SimHei',             # Windows 备选
    'Arial Unicode MS',   # 跨平台 Unicode
    'Noto Sans CJK SC',   # Linux 常见
    'DejaVu Sans',        # 最终回退
]
plt.rcParams['axes.unicode_minus'] = False

# 强制白底：避免在 dark-themed IDE 中运行时被自动切换为黑底
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['text.color'] = 'black'

# 项目根目录自动推断：优先基于 notebook 所在位置
_cwd = os.getcwd()
if os.path.exists(os.path.join(_cwd, '01_symbolic_feature_extraction.ipynb')):
    BASE_DIR = os.path.abspath(os.path.join(_cwd, '..', '..'))
else:
    # fallback：从当前路径向上搜索包含 CODE/datasets 的目录
    # 注意：用 os.path.dirname(_p) == _p 判断根目录，跨平台兼容（macOS 根为 '/'，Windows 根为 'C:\\' 等）
    _p = _cwd
    while not os.path.exists(os.path.join(_p, 'CODE', 'datasets')):
        _parent = os.path.dirname(_p)
        if _parent == _p:
            raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
        _p = _parent
    BASE_DIR = _p
MELODY_DIR = os.path.join(BASE_DIR, 'CODE', 'datasets', 'melodies')
# 对比曲目：瑶族舞曲
YAOZU_PATH = os.path.join(MELODY_DIR, '瑶族舞曲.midi')
# 图片输出目录锚定项目根，不随运行目录变化
FIGURES_DIR = os.path.join(BASE_DIR, 'CODE', 'chapter04', 'output_figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

MOLIHUA_PATH = os.path.join(MELODY_DIR, '茉莉花.midi')
print('BASE_DIR:', BASE_DIR)
print('YAOZU_PATH:', YAOZU_PATH)
print('FIGURES_DIR:', FIGURES_DIR)


## 0. 工具函数

封装 MIDI 解析与特征提取的常用函数，便于后续复用。


In [ ]:
def load_midi_notes(filepath, skip_drums=True, track_index=None):
    """从 MIDI 文件提取按 onset 排序的音符序列 [(pitch, onset, duration), ...]。"""
    pm = pretty_midi.PrettyMIDI(filepath)
    notes = []
    instruments = pm.instruments
    if track_index is not None:
        if (isinstance(track_index, bool) or not isinstance(track_index, int) or
                not 0 <= track_index < len(instruments)):
            raise ValueError('track_index 必须是现有轨道的非负整数索引。')
        instruments = [instruments[track_index]]
    for instrument in instruments:
        if skip_drums and instrument.is_drum:
            continue
        for note in instrument.notes:
            notes.append((note.pitch, note.start, note.end - note.start))
    notes.sort(key=lambda x: x[1])
    return notes


def pitch_histogram(notes, use_pitch_class=True):
    """音高直方图。use_pitch_class=True 时按 0-11 聚合。"""
    pitches = [p % 12 for p, _, _ in notes] if use_pitch_class else [p for p, _, _ in notes]
    return Counter(pitches)


def interval_histogram(notes, lo=None, hi=None):
    """相邻音符音程差的分布；给出 lo/hi 时才截断到闭区间。"""
    if (lo is None) != (hi is None):
        raise ValueError('lo 和 hi 必须同时给出或同时省略')
    if lo is not None and lo > hi:
        raise ValueError('lo 不能大于 hi。')
    pitches = [p for p, _, _ in notes]
    intervals = []
    for i in range(1, len(pitches)):
        d = pitches[i] - pitches[i - 1]
        if lo is None or lo <= d <= hi:
            intervals.append(d)
    return Counter(intervals)


def ioi_sequence(notes):
    """相邻音符 onset 差序列。"""
    onsets = [on for _, on, _ in notes]
    return [onsets[i] - onsets[i - 1] for i in range(1, len(onsets))]


def quantize_ioi(iois, beat_dur, bins=None):
    """将非负 IOI 量化到网格，返回以 beat_dur 为单位的比值。"""
    if isinstance(beat_dur, bool):
        raise ValueError('beat_dur 必须是有限正数。')
    try:
        beat_dur = float(beat_dur)
    except (TypeError, ValueError) as exc:
        raise ValueError('beat_dur 必须是有限正数。') from exc
    if not math.isfinite(beat_dur) or beat_dur <= 0:
        raise ValueError('beat_dur 必须是有限正数。')

    if bins is None:
        bins = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0]
    else:
        bins = list(bins)
    if not bins:
        raise ValueError('bins 不能为空。')
    try:
        bins = [float(bin_value) for bin_value in bins]
    except (TypeError, ValueError) as exc:
        raise ValueError('bins 必须只包含有限非负数。') from exc
    if any(not math.isfinite(bin_value) or bin_value < 0 for bin_value in bins):
        raise ValueError('bins 必须只包含有限非负数。')

    ioi_values = []
    for ioi in iois:
        if isinstance(ioi, bool):
            raise ValueError('IOI 必须是有限非负数。')
        try:
            ioi_value = float(ioi)
        except (TypeError, ValueError) as exc:
            raise ValueError('IOI 必须是有限非负数。') from exc
        if not math.isfinite(ioi_value) or ioi_value < 0:
            raise ValueError('IOI 必须是有限非负数。')
        ioi_values.append(ioi_value)

    ratios = [ioi / beat_dur for ioi in ioi_values]
    quantized = []
    for r in ratios:
        nearest = min(bins, key=lambda x: abs(x - r))
        quantized.append(nearest)
    return quantized


def parsons_code(notes):
    """Parsons Code: * = 首音, u = 上行, d = 下行, r = 重复。"""
    pitches = [p for p, _, _ in notes]
    if not pitches:
        return ''
    code = '*'
    for i in range(1, len(pitches)):
        if pitches[i] > pitches[i - 1]:
            code += 'u'
        elif pitches[i] < pitches[i - 1]:
            code += 'd'
        else:
            code += 'r'
    return code


# 小规模边界检查：正常量化、零/负拍长、空网格与非有限输入。
assert quantize_ioi([0.25, 0.50], 1.0, bins=[0.25, 0.5]) == [0.25, 0.5]
for bad_args in [
    ([0.25], 0.0, [0.25]),
    ([0.25], -1.0, [0.25]),
    ([0.25], 1.0, []),
    ([float('nan')], 1.0, [0.25]),
]:
    try:
        quantize_ioi(*bad_args)
    except ValueError:
        pass
    else:
        raise AssertionError(f'预期 ValueError，但输入通过：{bad_args}')
try:
    interval_histogram([(60, 0.0, 1.0), (62, 1.0, 1.0)], lo=2, hi=-2)
except ValueError:
    pass
else:
    raise AssertionError('lo > hi 时应抛出 ValueError')

print('工具函数与边界检查已加载')


## (一) 音高直方图

统计每个音级（pitch class，C=0, C#=1, ..., B=11）出现的频次，直接记录当前文件的音级使用比例；调式判断还需要中心音、句法位置与可靠谱源等证据。


In [ ]:
molihua_notes = load_midi_notes(MOLIHUA_PATH)
print(f'茉莉花：共 {len(molihua_notes)} 个音符')
print(f'音高范围：{min(p for p,_,_ in molihua_notes)} ~ {max(p for p,_,_ in molihua_notes)}')

molihua_pitch_hist = pitch_histogram(molihua_notes)
print('音级分布:', dict(sorted(molihua_pitch_hist.items())))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

pitch_classes = list(range(12))
counts = [molihua_pitch_hist.get(pc, 0) for pc in pitch_classes]
labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
colors = ['#2c3e50' if c > 0 else '#ecf0f1' for c in counts]

bars = ax.bar(labels, counts, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('音级')
ax.set_ylabel('音符数')
ax.set_title('《茉莉花》音高直方图')
ax.set_ylim(0, max(counts) * 1.15)

for bar, c in zip(bars, counts):
    if c > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(c), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_pitch_hist_molihua.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f'已保存：{out_path}')


### 对比分析：瑶族舞曲

将本地 `melodies` 目录中的《瑶族舞曲》MIDI 与《茉莉花》的音级分布作描述性对比。项目未附该文件的谱源、版本、编制与许可证说明，因此不依据文件名或直方图断定作品年代、编制或调式；这些判断需要可靠曲谱与音乐学来源。


In [ ]:
yaozu_notes = load_midi_notes(YAOZU_PATH)
print(f'瑶族舞曲：共 {len(yaozu_notes)} 个音符')
print(f'音高范围：{min(p for p,_,_ in yaozu_notes)} ~ {max(p for p,_,_ in yaozu_notes)}')

yaozu_pitch_hist = pitch_histogram(yaozu_notes)
print('音级分布:', dict(sorted(yaozu_pitch_hist.items())))


## (二) 音程向量

计算相邻音符的音程距离（半音数）。

**正文图题（`fig_interval_hist_molihua.png`）：《茉莉花》的音程分布直方图。**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

# 茉莉花
ax = axes[0]
m_counts = [molihua_pitch_hist.get(pc, 0) for pc in range(12)]
m_total = sum(m_counts)
m_norm = [c / m_total * 100 if m_total else 0 for c in m_counts]
colors_m = ['#2c3e50' if c > 0 else '#ecf0f1' for c in m_counts]
ax.bar(labels, m_norm, color=colors_m, edgecolor='black', linewidth=0.5)
ax.set_xlabel('音级')
ax.set_ylabel('占比（%）')
ax.set_title('《茉莉花》（五声音级）')
ax.set_ylim(0, max(m_norm) * 1.2)

# 瑶族舞曲
ax = axes[1]
b_counts = [yaozu_pitch_hist.get(pc, 0) for pc in range(12)]
b_total = sum(b_counts)
b_norm = [c / b_total * 100 if b_total else 0 for c in b_counts]
ax.bar(labels, b_norm, color='#bdc3c7', edgecolor='black', linewidth=0.5)
ax.set_xlabel('音级')
ax.set_ylabel('占比（%）')
ax.set_title('《瑶族舞曲》')
ax.set_ylim(0, max(b_norm) * 1.2)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_pitch_comparison.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f'已保存：{out_path}')


In [ ]:
molihua_iv_all = interval_histogram(molihua_notes)
molihua_iv = interval_histogram(molihua_notes, lo=-12, hi=12)
print('茉莉花音程分布（固定 25 维范围）:', dict(sorted(molihua_iv.items())))

total_iv = sum(molihua_iv_all.values())
molihua_outside = total_iv - sum(molihua_iv.values())
outside_detail = {k: v for k, v in sorted(molihua_iv_all.items()) if not -12 <= k <= 12}
print(f'全部相邻音程={total_iv}；±12 半音之外={molihua_outside}，明细={outside_detail}')
for k, v in sorted(molihua_iv.items()):
    print(f'{k:+3d} semitones: {v:3d} ({v/total_iv*100:.1f}%)')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
interval_range = list(range(-12, 13))
counts = [molihua_iv.get(i, 0) for i in interval_range]
colors = ['#2c3e50' if c > 0 else '#ecf0f1' for c in counts]

bars = ax.bar([str(i) for i in interval_range], counts, color=colors, edgecolor='black', linewidth=0.3)
ax.set_xlabel('音程（半音）')
ax.set_ylabel('次数')
ax.set_ylim(0, max(counts) + 3)
ax.text(0.99, 0.96, f'±12 半音范围外：{molihua_outside}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10)
plt.setp(ax.get_xticklabels(), rotation=45)

for bar, c in zip(bars, counts):
    if c > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(c), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_interval_hist_molihua.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f'已保存：{out_path}')


In [ ]:
yaozu_iv_all = interval_histogram(yaozu_notes)
yaozu_iv = interval_histogram(yaozu_notes, lo=-12, hi=12)
yaozu_outside = sum(yaozu_iv_all.values()) - sum(yaozu_iv.values())
print('瑶族舞曲 音程分布 (top 10):', yaozu_iv_all.most_common(10))
print(f'全部相邻音程={sum(yaozu_iv_all.values())}；±12 半音之外={yaozu_outside}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# 自适应音程范围：取两首乐曲音程的并集，避免截断
all_keys = set(molihua_iv_all.keys()) | set(yaozu_iv_all.keys())
iv_min, iv_max = min(all_keys), max(all_keys)
iv_range = list(range(iv_min, iv_max + 1))

ax = axes[0]
m_counts = [molihua_iv_all.get(i, 0) for i in iv_range]
m_total = sum(m_counts)
m_norm = [c / m_total * 100 if m_total else 0 for c in m_counts]
colors_m = ['#2c3e50' if abs(i) in (2, 3) else '#bdc3c7' if c > 0 else '#ecf0f1'
            for i, c in zip(iv_range, m_counts)]
ax.bar([str(i) for i in iv_range], m_norm, color=colors_m, edgecolor='black', linewidth=0.3)
ax.set_xlabel('音程（半音）')
ax.set_ylabel('占比（%）')
ax.set_title('《茉莉花》')
plt.setp(ax.get_xticklabels(), rotation=45)

ax = axes[1]
b_counts = [yaozu_iv_all.get(i, 0) for i in iv_range]
b_total = sum(b_counts)
b_norm = [c / b_total * 100 if b_total else 0 for c in b_counts]
# 瑶族舞曲：同样高亮主要音程 ±2, ±3
highlight_yaozu = {2, 3, -2, -3}
colors_b = ['#2c3e50' if i in highlight_yaozu and c > 0 else '#bdc3c7' if c > 0 else '#ecf0f1'
            for i, c in zip(iv_range, b_counts)]
ax.bar([str(i) for i in iv_range], b_norm, color=colors_b, edgecolor='black', linewidth=0.3)
ax.set_xlabel('音程（半音）')
ax.set_ylabel('占比（%）')
ax.set_title('《瑶族舞曲》')
plt.setp(ax.get_xticklabels(), rotation=45)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_interval_comparison.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f'已保存：{out_path}')


## (三) 节奏模式

提取相邻音符的 IOI（Inter-Onset Interval）。本地《茉莉花》文件为 2/4 拍，因此一个四分音符时值等于一拍；代码按 tempo map 换算 IOI。MIDI 格式本身可以保存演奏时序；这里只能确认该文件的 onset 落在规则网格上，不能据此概括所有 MIDI 或推断原演奏是否有自由伸缩（rubato）。


In [ ]:
molihua_iois = ioi_sequence(molihua_notes)
print(f'IOI 数量：{len(molihua_iois)}')
print(f'IOI 范围：{min(molihua_iois):.3f}s ~ {max(molihua_iois):.3f}s')

ioi_counter = Counter(round(ioi, 3) for ioi in molihua_iois)
most_common_ioi = ioi_counter.most_common(1)[0][0]
molihua_pm = pretty_midi.PrettyMIDI(MOLIHUA_PATH)
tempo_times, tempi = molihua_pm.get_tempo_changes()
if len(tempi) != 1:
    raise ValueError('本例量化代码假设全曲只有一个速度；检测到速度变化，请按 tempo map 分段换算。')
molihua_tempo_bpm = float(tempi[0])
molihua_quarter_dur = 60.0 / molihua_tempo_bpm
print(f'最频繁 IOI：{most_common_ioi:.3f}s')
print(f'MIDI 速度：{molihua_tempo_bpm:.1f} BPM；四分音符时值：{molihua_quarter_dur:.3f}s')
print(f'最频繁 IOI = {most_common_ioi / molihua_quarter_dur:.2f} 个四分音符拍')

# 2/4 拍中分母 4 表示以四分音符为一拍；不能由最频繁 IOI 反推 BPM。
molihua_beat_dur = molihua_quarter_dur
# 同理：quantized_iois 避免与 quantize_ioi() 内部 quantized 局部变量遮蔽
quantized_iois = quantize_ioi(molihua_iois, molihua_beat_dur)
q_counter = Counter(quantized_iois)
print('量化后节奏值分布:', dict(sorted(q_counter.items())))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.hist(molihua_iois, bins=30, color='#bdc3c7', edgecolor='black', linewidth=0.3)
ax.axvline(molihua_beat_dur, color='black', linestyle='--', label=f'四分音符时值 = {molihua_beat_dur:.3f}s')
ax.set_xlabel('起奏间隔（秒）')
ax.set_ylabel('次数')
ax.set_title('《茉莉花》原始 IOI 分布')
ax.legend()

ax = axes[1]
labels_q = [f'{v:g}' for v in sorted(q_counter.keys())]
counts_q = [q_counter[k] for k in sorted(q_counter.keys())]
ax.bar(labels_q, counts_q, color='#2c3e50', edgecolor='black', linewidth=0.3)
ax.set_xlabel('四分音符时值数')
ax.set_ylabel('次数')
ax.set_title('量化后 IOI 分布')

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_rhythm_ioi.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f'已保存：{out_path}')

## (四) 旋律轮廓

### 1. Parsons Code
用方向符号描述旋律走向：`*`（首音）、`u`（上行）、`d`（下行）、`r`（重复）。


In [ ]:
molihua_parsons = parsons_code(molihua_notes)
print(f'茉莉花 Parsons Code（前50符）：{molihua_parsons[:50]}')
print(f'总长度：{len(molihua_parsons)}')
print('轮廓符号分布:', dict(Counter(molihua_parsons)))


### 2. 旋律轮廓
以时间为横轴、音高为纵轴，直观展示旋律的起伏形态。

**正文图题：《茉莉花》的旋律轮廓（MIDI 音高随时间变化）。**


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

melody_onsets = [on for _, on, _ in molihua_notes]
melody_pitches = [p for p, _, _ in molihua_notes]

ax.plot(melody_onsets, melody_pitches, marker='o', markersize=2, linewidth=0.8, color='#2c3e50')
ax.fill_between(melody_onsets, melody_pitches, alpha=0.15, color='#2c3e50')

top_y = max(melody_pitches) + 8
for i in range(0, len(melody_onsets), 10):
    name = pretty_midi.note_number_to_name(melody_pitches[i])
    ax.annotate(name,
                xy=(melody_onsets[i], melody_pitches[i]),
                xytext=(melody_onsets[i], top_y),
                textcoords='data',
                ha='center', fontsize=7, color='#000000',
                arrowprops=dict(arrowstyle='-', color='gray',
                                lw=0.5, ls='--'))

ax.set_xlabel('时间（秒）')
ax.set_ylabel('MIDI 音高')
ax.set_ylim(min(melody_pitches) - 3, max(melody_pitches) + 12)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'fig_melodic_contour.png')
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f'已保存：{out_path}')

## 汇总

| 特征                   | 输出文件                              |
|----------------------|-----------------------------------|
| 音高直方图（茉莉花）           | `fig_pitch_hist_molihua.png`      |
| 音高直方图（对比）            | `fig_pitch_comparison.png`        |
| 音程分布（茉莉花）            | `fig_interval_hist_molihua.png`   |
| 音程分布（对比）             | `fig_interval_comparison.png`     |
| IOI                  | `fig_rhythm_ioi.png`              |
| 旋律轮廓                 | `fig_melodic_contour.png`         |
